In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy.ndimage import gaussian_filter
from matplotlib.patches import Rectangle, Circle
import io
import os

# ==========================================
# 1. CONFIGURATION
# ==========================================
CSV_FILENAME = 'D:/Data/1.csv'
STOP_AT_NO = 10*60*25
SIGMA = 5.0
RESOLUTION = 500

# Main Arena Boundaries
ARENA_LEFT = 322
ARENA_TOP = 9      
ARENA_WIDTH = 162
ARENA_HEIGHT = 464

# Define the specific Zones
zones = {
    "Subordinate Sniffing": {"type": "circle", "left": 349, "top": 65, "width": 100, "height": 100, "color": "lime"},
    "Dominant Chamber": {"type": "rect", "left": 322, "top": 327, "width": 159, "height": 150, "color": "yellow"},
    "Subordinate Chamber": {"type": "rect", "left": 326, "top": 20, "width": 159, "height": 147, "color": "cyan"},
    "Dominant Sniffing": {"type": "circle", "left": 351, "top": 339, "width": 100, "height": 100, "color": "magenta"}
}

# ==========================================
# 2. LOAD DATA
# ==========================================
sample_data = """No.,Date,Time,Head X,Head Y,Body X,Body Y,Tail X,Tail Y
1,2026/6/8,20:25:16,396,353,394,359,397,365
2,2026/6/8,20:25:16,389,356,395,359,397,366
3,2026/6/8,20:25:16,389,356,395,360,398,367
4,2026/6/8,20:25:16,389,356,395,358,397,363
5,2026/6/8,20:25:16,393,340,387,341,383,346
6,2026/6/8,20:25:16,390,355,397,357,402,361
7,2026/6/8,20:25:16,405,362,400,357,391,354
8,2026/6/8,20:25:16,408,363,402,357,391,354
9,2026/6/8,20:25:16,412,364,404,358,392,354
10,2026/6/8,20:25:17,416,367,408,359,394,355
11,2026/6/8,20:25:17,420,370,411,360,397,353
12,2026/6/8,20:25:17,401,353,414,360,423,370
13,2026/6/8,20:25:17,403,354,416,361,426,370
14,2026/6/8,20:25:17,406,356,420,362,429,375
15,2026/6/8,20:25:17,413,353,424,365,433,379
16,2026/6/8,20:25:17,417,355,427,368,435,383
17,2026/6/8,20:25:17,421,357,429,372,437,387
18,2026/6/8,20:25:17,424,360,430,374,435,389
19,2026/6/8,20:25:17,428,362,431,376,435,390
20,2026/6/8,20:25:17,432,363,432,378,431,395
21,2026/6/8,20:25:17,434,365,433,379,430,396"""

if os.path.exists(CSV_FILENAME):
    print(f"Loading data from '{CSV_FILENAME}'...")
    df = pd.read_csv(CSV_FILENAME)
else:
    print(f"File '{CSV_FILENAME}' not found. Using embedded sample data.")
    df = pd.read_csv(io.StringIO(sample_data.strip()))

# ==========================================
# 3. FILTER DATA
# ==========================================
if STOP_AT_NO is not None:
    df = df[df['No.'] <= STOP_AT_NO].copy()

ARENA_RIGHT = ARENA_LEFT + ARENA_WIDTH
ARENA_BOTTOM = ARENA_TOP + ARENA_HEIGHT

df = df[(df['Body X'] >= ARENA_LEFT) & 
        (df['Body X'] <= ARENA_RIGHT) & 
        (df['Body Y'] >= ARENA_TOP) & 
        (df['Body Y'] <= ARENA_BOTTOM)].copy()

if df.empty:
    print("Error: No data found within arena boundaries.")
    exit()

print(f"Processing {len(df)} rows...")

x_coords = df['Body X'].values
y_coords = df['Body Y'].values

# ==========================================
# 4. GENERATE GAUSSIAN HEATMAP
# ==========================================
padding = 20
x_min, x_max = ARENA_LEFT - padding, ARENA_RIGHT + padding
y_min, y_max = ARENA_TOP - padding, ARENA_BOTTOM + padding

# Create histogram
heatmap, xedges, yedges = np.histogram2d(
    x_coords, y_coords, 
    bins=RESOLUTION, 
    range=[[x_min, x_max], [y_min, y_max]]
)

# Apply Gaussian blur
bin_size_x = (x_max - x_min) / RESOLUTION
bin_size_y = (y_max - y_min) / RESOLUTION
avg_bin_size = (bin_size_x + bin_size_y) / 2.0
sigma_bins = SIGMA / avg_bin_size

heatmap = gaussian_filter(heatmap, sigma=sigma_bins)

# *** THE FIX: Transpose the heatmap so X is horizontal and Y is vertical ***
heatmap = heatmap.T 

if heatmap.max() > 0:
    heatmap = heatmap / heatmap.max()

# ==========================================
# 5. PLOTTING
# ==========================================
colors = ['darkblue', 'blue', 'cyan', 'yellow', 'orange', 'red']
custom_cmap = mcolors.LinearSegmentedColormap.from_list('blue_to_red', colors)

# Adjust figsize based on arena orientation
if ARENA_WIDTH > ARENA_HEIGHT:
    fig_size = (12, 6)
else:
    fig_size = (6, 12)

plt.figure(figsize=fig_size, facecolor='darkblue')
ax = plt.gca()
ax.set_facecolor('darkblue')

# Plot the transposed heatmap. 
# We use origin='lower' so Y goes up (standard math), matching the zones.
plt.imshow(heatmap, extent=[xedges[0], xedges[-1], yedges[0], yedges[-1]], 
           origin='lower', cmap=custom_cmap, aspect='equal')

# Draw Main Arena
arena_rect = Rectangle(
    (ARENA_LEFT, ARENA_TOP), ARENA_WIDTH, ARENA_HEIGHT,
    linewidth=2, edgecolor='white', facecolor='none',
    linestyle='--', label='Main Arena', zorder=10
)
ax.add_patch(arena_rect)

# Draw Zones
for name, props in zones.items():
    if props["type"] == "rect":
        patch = Rectangle(
            (props["left"], props["top"]), props["width"], props["height"],
            linewidth=2, edgecolor=props["color"], facecolor='none',
            linestyle='-', label=name, zorder=11
        )
    elif props["type"] == "circle":
        cx = props["left"] + (props["width"] / 2)
        cy = props["top"] + (props["height"] / 2)
        radius = props["width"] / 2
        patch = Circle(
            (cx, cy), radius,
            linewidth=2, edgecolor=props["color"], facecolor='none',
            linestyle='-', label=name, zorder=11
        )
    ax.add_patch(patch)

# Formatting
title_suffix = f" (No. 1 to {STOP_AT_NO})" if STOP_AT_NO is not None else " (All Data)"
plt.title(f'Gaussian Heatmap{title_suffix}', fontsize=14, color='white')
plt.xlabel('Body X', fontsize=12, color='white')
plt.ylabel('Body Y', fontsize=12, color='white')

plt.xlim(x_min, x_max)
plt.ylim(y_min, y_max)

cbar = plt.colorbar(label='Normalized Density', shrink=0.5, ax=ax, location='left')
cbar.ax.yaxis.set_tick_params(color='white')
cbar.outline.set_edgecolor('white')
plt.setp(plt.getp(cbar.ax.axes, 'yticklabels'), color='white')

handles, labels = ax.get_legend_handles_labels()
by_label = dict(zip(labels, handles))
plt.legend(by_label.values(), by_label.keys(), facecolor='darkblue', edgecolor='white', labelcolor='white', loc='center left', bbox_to_anchor=(1, 0.5))

plt.grid(True, linestyle='--', alpha=0.3, color='white')
plt.tight_layout()
plt.savefig('body_heatmap_aligned.png', dpi=300, facecolor='white')
print("Heatmap saved as 'body_heatmap_aligned.png'")
plt.show()